# 00 — Data setup & sanity
Shared analysis table `fe` = `features ⨝ era5`, |lat|<40°, both missions, materialized at `/data/scratch/a/snesbitt/pf_db/analysis/shear_cape`. Helpers live in `_shc.py`.

**Kernel:** `Python (pf)` for 00–05. **Predictors:** ambient CAPE = `p90_cape_2p50deg` (pre-convective box, not the depleted centroid); shear = `shear_6000m_centroid` (0–6 km). See `../shear_cape_hypotheses_PLAN.md`.

In [1]:
import sys; sys.path.insert(0, '.')
from _shc import *
con = connect()


In [2]:
print('columns:', len(con.execute('DESCRIBE fe').df()))
q(con, '''SELECT mission, surf, count(*) n,
      count(*) FILTER(WHERE ht40_km>0) deep_40dbz,
      round(avg(shear_6000m_centroid),1) sh6, round(avg(p90_cape_2p50deg),0) cape
   FROM fe GROUP BY 1,2 ORDER BY 1,2''')

columns: 47


,mission,surf,n,deep_40dbz,sh6,cape
0,GPM,land,5416783,738440,8.5,819.0
1,GPM,ocean,31826968,1889752,10.8,448.0
2,TRMM,land,13410714,2076886,9.0,851.0
3,TRMM,ocean,73366114,5679882,11.2,486.0


### Predictor ranges & the CAPE–shear anticorrelation (why we must condition)

In [3]:
q(con, f'''SELECT mission, surf,
      corr({AMBIENT_CAPE}, {SHEAR}) AS cape_shear_corr,
      quantile_cont({AMBIENT_CAPE},0.5) med_cape, quantile_cont({SHEAR},0.5) med_shear
   FROM fe GROUP BY 1,2 ORDER BY 1,2''')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,mission,surf,cape_shear_corr,med_cape,med_shear
0,GPM,land,-0.299460,608.281738,6.381055
1,GPM,ocean,-0.377068,240.712891,8.280320
2,TRMM,land,-0.333366,651.117310,6.543093
3,TRMM,ocean,-0.402802,297.807129,8.423089
